# IBM SpaceX Capstone — reproducible exploratory analysis
Source: publicly archived IBM course dataset; selected columns transcribed and checked against 90 rows. This notebook operates on the included derived CSV.

In [ ]:
import pandas as pd, sqlite3
df=pd.read_csv('spaceX_analysis_90_launches.csv')
print(df.shape)
df.head()

In [ ]:
print(df.groupby('LaunchSite')['Class'].agg(['count','sum','mean']))
print(df.groupby('Year')['Class'].agg(['count','sum','mean']))

In [ ]:
con=sqlite3.connect(':memory:');df.to_sql('spacex',con,index=False)
print(pd.read_sql_query('SELECT LaunchSite,COUNT(*) launches,ROUND(AVG(Class)*100,1) success_pct FROM spacex GROUP BY LaunchSite',con))

In [ ]:
import matplotlib.pyplot as plt
df.groupby('Year')['Class'].mean().mul(100).plot(marker='o',title='Annual recovery success (%)');plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
X=df[['FlightNumber','PayloadMass','Orbit','LaunchSite']];y=df['Class']
pre=ColumnTransformer([('num',StandardScaler(),['FlightNumber','PayloadMass']),('cat',OneHotEncoder(handle_unknown='ignore'),['Orbit','LaunchSite'])])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
p=Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=1500))]).fit(Xtr,ytr)
print(classification_report(yte,p.predict(Xte)))